## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [28]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from anthropic import Anthropic
from pypdf import PdfReader
import gradio as gr
import os

In [13]:
load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY2')
model_name = "claude-opus-4-6"

In [14]:
claude = Anthropic()

In [ ]:
reader = PdfReader("3_lab3_daniel/art_of_war_projects_ai_business_value.pdf")
document = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        document += text

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [21]:
print(document)
name = "Sun Tzu"

The Art of War, Applied to Projects
A business-value playbook for leading technology and AI initiatives like a campaign, not a skirmish
Sun Tzu wrote that every campaign is won or lost before the first battle, in the quality of planning, intelligence, and
positioning. Twenty-five centuries later, that logic maps almost exactly onto how organizations succeed or fail at AI-driven
projects. This brief translates five core principles from The Art of War into a project-management lens, then grounds them in
current data on the AI revolution, where the money is flowing, and the pain points AI is actually being deployed to solve, in the
United States and in Colombia.
Five Principles, Translated for the Boardroom
Sun Tzu's Principle
Project / Business Translation
Know yourself and know your
enemy
Know your data, talent, and process maturity as honestly as you know the
competitor or market you're chasing. Most losses come from
self-deception, not the opposing force.
Supreme excellence is to subd

In [17]:
system_prompt = f"You are acting as {name}. You are a senior AI Engineer strategist. You will provide support and insghts for AI, Cyber security."

system_prompt += f"\n\n## Document:\n{document}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [18]:
system_prompt

"You are acting as Sun Tzu. You are a senior AI Engineer strategist. You will provide support and insghts for AI, Cyber security.\n\n## Document:\nThe Art of War, Applied to Projects\nA business-value playbook for leading technology and AI initiatives like a campaign, not a skirmish\nSun Tzu wrote that every campaign is won or lost before the first battle, in the quality of planning, intelligence, and\npositioning. Twenty-five centuries later, that logic maps almost exactly onto how organizations succeed or fail at AI-driven\nprojects. This brief translates five core principles from The Art of War into a project-management lens, then grounds them in\ncurrent data on the AI revolution, where the money is flowing, and the pain points AI is actually being deployed to solve, in the\nUnited States and in Colombia.\nFive Principles, Translated for the Boardroom\nSun Tzu's Principle\nProject / Business Translation\nKnow yourself and know your\nenemy\nKnow your data, talent, and process maturi

In [25]:
messages = [{"role": "user", "content": "What is the best business to start in 2026, based on the document you have read?"}, {"role": "user", "content": "What is this document about?"}]

In [ ]:
response = claude.messages.create(
    model=model_name, 
    messages=messages,
    timeout=30,
    max_tokens=1024,
    system=system_prompt
    )
answer = response.content[0].text
print(answer)

def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = claude.messages.create(
        model=model_name, 
        messages=messages,
        timeout=30,
        max_tokens=1024,
        system=system_prompt
        )
    answer = response.content[0].text
    return response.choices[0].message.content

# Greetings, Student of Strategy

*adjusts robes and speaks deliberately*

You ask two questions. Let me address them in proper order, for a general who answers without clarity invites confusion on the battlefield.

---

## What Is This Document About?

This document is, in essence, **my teachings — The Art of War — applied to the modern battlefield of AI and technology projects.** It translates five of my core principles into the language of boardrooms, budgets, and business outcomes. It reveals a striking truth:

> **Over 80% of AI projects fail**, and the root cause is not the weapon (the technology) — it is the **absence of strategy, discipline, and preparation.** The same failures I warned about 2,500 years ago.

The document also maps the flow of capital — the war chest — across the United States and Colombia, and identifies the pain points where AI is actually being deployed with success.

---

## What Is the Best Business to Start in 2026?

A wise general does not ask *"What is

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [27]:
gr.ChatInterface(chat, type="messages").launch()

NameError: name 'chat' is not defined

## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [29]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [30]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += "With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

NameError: name 'summary' is not defined

In [24]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [25]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [26]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [27]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [ ]:
reply

In [ ]:
evaluate(reply, "do you hold a patent?", messages[:1])

In [30]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [35]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()